# JSONBench walkthrough

Generate a small travel dataset in memory.

In [ ]:
from json_bench.data import generate_travel_dataset

dataset = generate_travel_dataset(
    scale=0.01,
    seed=23,
    reference_date="2025-11-01",
)

## Deploy the databases

Start Docker and run `uv sync --locked` first. BONDI waits for each database to be ready.
Couchbase uses Community Edition with a `travel` bucket. Change the names or ports if needed.

In [ ]:
from bondi.deployment import deploy_couchbase, deploy_mongodb

mongo_deployment = deploy_mongodb(name="jsonbench-travel", port=27127)
couchbase_deployment = deploy_couchbase(
    name="jsonbench-travel-cb",
    management_port=18291,
    query_port=18293,
    kv_port=21230,
    bucket="travel",
)
print(mongo_deployment.uri)
print(couchbase_deployment.console_url)

## Load the same dataset into both databases

Load each collection into MongoDB and Couchbase, preserving document IDs and date strings.
BONDI derives Couchbase keys from the IDs and creates no query indexes.
The default `create` mode requires empty collections. Run cleanup before repeating deployment and loading.

In [ ]:
from bondi.couchbase import CouchbaseCollectionRef
from bondi.data import load_documents
from bondi.mongodb import MongoCollectionRef

for name, documents in dataset.items():
    mongo_result = load_documents(
        documents,
        target=mongo_deployment,
        collection=MongoCollectionRef("travel", name),
    )
    couchbase_result = load_documents(
        documents,
        target=couchbase_deployment,
        collection=CouchbaseCollectionRef("travel", "_default", name),
    )
    print(mongo_result.destinations, couchbase_result.destinations)

## Translate Q1

Group all bookings by tour ID and sum their stored total prices.
These functions return BONDI query objects without executing them.

In [ ]:
from pprint import pprint

from json_bench.queries import translate_couchbase, translate_mongodb

q1 = {
    "name": "Query 1",
    "description": "Total booking value per tour",
    "primary_collection": "bookings",
    "operations": {"group": {"by": "tour_id", "aggregates": {"total_revenue": "sum(total_price)"}}},
}

mongo_query = translate_mongodb(q1, database="travel")
couchbase_query = translate_couchbase(q1, bucket="travel", scope="_default")

pprint(mongo_query.pipeline)
print(couchbase_query.statement)
print(couchbase_query.query_context)

## Execute Q1

Use BONDI executors to run the queries and retain their result documents.
Create the bookings index used by corrected JSONBench. Index creation is separate from translation.
For this correctness check, Couchbase waits for indexing to catch up with the loaded data.
This cell is not a performance measurement.

In [ ]:
from bondi.couchbase import CouchbaseExecutor, CouchbaseQuery
from bondi.mongodb import MongoExecutor
from couchbase.n1ql import QueryScanConsistency

with MongoExecutor.from_uri(mongo_deployment.uri, timeout_seconds=60) as executor:
    mongo_result = executor.run(mongo_query, collect_results=True)

with CouchbaseExecutor.from_uri(
    couchbase_deployment.uri,
    username=couchbase_deployment.username,
    password=couchbase_deployment.password,
    timeout_seconds=60,
) as executor:
    executor.run(
        CouchbaseQuery(
            "CREATE INDEX `bookings-tour_id` IF NOT EXISTS ON `bookings`(`tour_id`);",
            query_context=couchbase_query.query_context,
        )
    )
    couchbase_result = executor.run(
        couchbase_query,
        collect_results=True,
        query_kwargs={"scan_consistency": QueryScanConsistency.REQUEST_PLUS},
    )

pprint(mongo_result.documents)
pprint(couchbase_result.documents)

## Compare Q1 results

Match MongoDB's `_id` with Couchbase's `tour_id`, ignoring row order.
Require the same tour IDs and compare sums with a small floating-point tolerance.
Duplicate groups or empty results fail the check.

In [ ]:
from math import isclose

mongo_totals = {row["_id"]: row["total_revenue"] for row in mongo_result.documents}
couchbase_totals = {row["tour_id"]: row["total_revenue"] for row in couchbase_result.documents}

assert len(mongo_totals) == len(mongo_result.documents), "Duplicate MongoDB groups"
assert len(couchbase_totals) == len(couchbase_result.documents), "Duplicate Couchbase groups"
assert mongo_totals, "Q1 returned no tour groups"
assert mongo_totals.keys() == couchbase_totals.keys(), "Tour IDs differ"

for tour_id, total in mongo_totals.items():
    assert isclose(total, couchbase_totals[tour_id], rel_tol=1e-9, abs_tol=1e-6), (
        f"Revenue differs for {tour_id}: {total} vs {couchbase_totals[tour_id]}"
    )

print(f"Q1 results match for all {len(mongo_totals)} tours.")

## Clean up when finished

Remove both deployments, including their data and saved configuration.

In [ ]:
mongo_deployment.remove()
couchbase_deployment.remove()